# Retriever & Reranker Comparison
### One Shared Evaluation Setup -- Multiple Strategies in Haystack / LangChain Style

**NLP / Information Retrieval Portfolio**

This notebook runs a rigorous, apples-to-apples comparison of **six retrieval strategies**
and **four reranking approaches** against a single shared evaluation harness.

Every strategy receives:
- The **same indexed corpus** (150+ document chunks from 30 technical articles)
- The **same 40 labelled test queries** with ground-truth relevance judgements
- The **same evaluation metrics**: Recall@K, MRR, nDCG@K, Precision@K, Latency

| # | Retrieval Strategy | Type |
|---|---|---|
| 1 | BM25 | Sparse lexical |
| 2 | TF-IDF Cosine | Sparse vector |
| 3 | Dense Embedding | Dense semantic |
| 4 | ColBERT MaxSim | Late interaction |
| 5 | Hybrid BM25+Dense (RRF) | Hybrid fusion |
| 6 | Hybrid BM25+Dense (Linear) | Hybrid fusion |

| # | Reranking Strategy | Type |
|---|---|---|
| 1 | Cross-encoder score | Pointwise |
| 2 | MonoT5-style reranker | Generative |
| 3 | Cohere-style relevance | API-based simulation |
| 4 | Diversity reranker (MMR) | Diversity-aware |

## What Are Retrievers and Rerankers?

Modern RAG pipelines use a **two-stage** retrieval approach:

```
Stage 1: RETRIEVER (fast, broad)
  Query --> [Retriever] --> top-100 candidates
                           (cheap: exact match or ANN search)

Stage 2: RERANKER (slow, accurate)
  Query + top-100 --> [Reranker] --> top-5 final results
                                    (expensive: cross-attention or LLM scoring)
```

### Why Two Stages?

| Concern | Retriever | Reranker |
|---|---|---|
| **Speed** | Must support 1000+ QPS | Can be 100x slower |
| **Coverage** | Must not miss relevant docs | Fine-grained selection |
| **Cost** | Cheap (inverted index / ANN) | Expensive (neural model per pair) |
| **Accuracy** | Recall-focused | Precision-focused |

### Framework Mapping

**Haystack two-stage pipeline:**
```python
pipe.add_component('retriever', InMemoryBM25Retriever(store, top_k=50))
pipe.add_component('reranker', TransformersSimilarityRanker(top_k=5))
pipe.connect('retriever.documents', 'reranker.documents')
```

**LangChain equivalent:**
```python
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
retriever = ContextualCompressionRetriever(
    base_retriever=vectorstore.as_retriever(search_kwargs={'k': 50}),
    doc_compressor=CrossEncoderReranker(model=ce_model, top_n=5)
)
```

## Learning Objectives

1. Build a **shared evaluation harness** that is completely retriever/reranker agnostic
2. Implement all 6 retrieval strategies from scratch, understanding each algorithm
3. Implement all 4 reranking strategies and understand when each is appropriate
4. Run a **fair comparison** -- same corpus, same queries, same metrics for every combination
5. Produce a **decision matrix** recommending the best strategy per query type and latency budget
6. Understand the **recall-precision trade-off** that motivates two-stage retrieval

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ['pandas', 'numpy', 'matplotlib', 'seaborn', 'plotly']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Ready.')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import re, math, time, copy, hashlib
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 100)
plt.rcParams['figure.figsize'] = (14, 5)
sns.set_style('whitegrid')
np.random.seed(42)
print('Imports OK')

## Shared Data Model

All strategies operate on the same `Document` type. The corpus is pre-processed once
and reused by every retriever -- ensuring a completely fair comparison.

In [ ]:
@dataclass
class Document:
    id: str = ''
    content: str = ''
    meta: dict = field(default_factory=dict)
    embedding: Optional[np.ndarray] = None
    score: Optional[float] = None

    def __post_init__(self):
        if not self.id:
            self.id = hashlib.sha256(self.content.encode()).hexdigest()[:12]

    def __repr__(self):
        return (f'Document(id={self.id}, len={len(self.content)}, '
                f'cat={self.meta.get("category","?")}, score={self.score})')


print('Document model defined.')

## Shared Corpus

30 source articles across 6 domains, split into overlapping chunks of 3 sentences.
Every retriever indexes **exactly the same chunks**.

In [ ]:
ARTICLES = [
    # Machine Learning
    {'title': 'Gradient Descent Optimisation', 'category': 'ml', 'content': (
        'Gradient descent minimises a loss function by iteratively moving in the direction '
        'of the negative gradient. Stochastic gradient descent (SGD) updates weights on '
        'each sample, making it noisy but fast. Mini-batch SGD uses batches of 32-256 '
        'samples, balancing noise and speed. Momentum accumulates velocity in the gradient '
        'direction, dampening oscillations. Nesterov accelerated gradient looks ahead before '
        'computing the gradient. Adaptive methods like Adam, RMSProp, and Adagrad maintain '
        'per-parameter learning rates. Adam combines momentum and adaptive rates: '
        'm_t = beta1*m_{t-1}+(1-beta1)*g_t, v_t = beta2*v_{t-1}+(1-beta2)*g_t^2. '
        'Learning rate schedules include cosine annealing, warmup, and step decay.'
    )},
    {'title': 'Regularisation Techniques', 'category': 'ml', 'content': (
        'Regularisation prevents overfitting by adding constraints or penalties to the loss. '
        'L1 (Lasso) adds the sum of absolute weights, encouraging sparsity and feature '
        'selection. L2 (Ridge) adds the sum of squared weights, shrinking all weights '
        'proportionally. Elastic Net combines L1 and L2. Dropout randomly zeros units '
        'during training at rate p (typically 0.1-0.5), acting as an ensemble of '
        'exponentially many networks. Batch normalisation normalises layer activations, '
        'reducing internal covariate shift and allowing higher learning rates. '
        'Early stopping halts training when validation loss stops improving.'
    )},
    {'title': 'Transformer Architecture', 'category': 'ml', 'content': (
        'Transformers use self-attention to model relationships between all positions in a '
        'sequence simultaneously. Multi-head attention projects queries Q, keys K, values V '
        'into h subspaces: Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) V. '
        'Positional encodings inject sequence order since attention is permutation invariant. '
        'The encoder stack processes the input; the decoder stack generates output with '
        'cross-attention to encoder representations. Pre-training variants include masked '
        'language modelling (BERT), causal LM (GPT), and sequence-to-sequence (T5, BART). '
        'Efficient variants: Longformer (sparse attention), Flash Attention (IO-aware).'
    )},
    {'title': 'Cross-Validation and Model Selection', 'category': 'ml', 'content': (
        'K-fold cross-validation splits data into k folds, training on k-1 and evaluating '
        'on the held-out fold, rotating until all folds serve as validation sets. '
        'Stratified k-fold preserves class proportions in each fold. '
        'Nested cross-validation separates hyperparameter search from final evaluation '
        'to avoid optimistic bias. Pipeline-aware CV applies preprocessing within each '
        'fold to prevent data leakage. TimeSeriesSplit respects temporal order.'
    )},
    {'title': 'Ensemble Methods', 'category': 'ml', 'content': (
        'Bagging trains multiple models on bootstrap samples and aggregates by voting or '
        'averaging. Random Forest extends bagging with random feature subsets at each split. '
        'Boosting trains models sequentially: AdaBoost reweights samples, Gradient Boosting '
        'fits residuals, and XGBoost/LightGBM/CatBoost are highly optimised implementations. '
        'Stacking trains a meta-learner on base model predictions from a holdout set. '
        'Diverse base models improve ensemble performance significantly.'
    )},
    # NLP
    {'title': 'Text Tokenisation Strategies', 'category': 'nlp', 'content': (
        'Word tokenisation splits on whitespace and punctuation but struggles with '
        'out-of-vocabulary words. Character-level models handle any string but produce '
        'very long sequences. Subword methods balance both approaches: BPE (Byte Pair '
        'Encoding) iteratively merges the most frequent adjacent token pairs. WordPiece '
        '(BERT) maximises training data likelihood of the vocabulary. Unigram language '
        'model (SentencePiece) uses a probabilistic approach. Typical vocabulary sizes '
        'range from 32k to 100k tokens.'
    )},
    {'title': 'Named Entity Recognition', 'category': 'nlp', 'content': (
        'NER identifies and classifies named entities: PERSON, ORG, GPE, DATE, MONEY. '
        'Approaches range from rule-based gazetteers to CRF with hand-crafted features '
        'to fine-tuned transformers (BERT-NER, spaCy transformers). IOB tagging marks '
        'B (begin), I (inside), O (outside) to identify entity spans. Nested NER handles '
        'overlapping entities. Evaluation uses entity-level F1 requiring exact span match.'
    )},
    {'title': 'Sentence Embeddings and Semantic Search', 'category': 'nlp', 'content': (
        'Sentence embeddings represent sentences as dense fixed-size vectors. SBERT '
        '(Sentence-BERT) fine-tunes BERT using siamese networks and NLI/STS tasks, '
        'enabling fast cosine similarity search. Contrastive learning (SimCSE) uses '
        'in-batch negatives for training. Approximate nearest neighbour (ANN) search '
        'scales to billions of documents using HNSW or IVF-PQ (FAISS library). '
        'MTEB (Massive Text Embedding Benchmark) evaluates across 58 tasks.'
    )},
    {'title': 'Topic Modelling', 'category': 'nlp', 'content': (
        'Topic modelling discovers latent thematic structure in document collections. '
        'LDA assumes documents are mixtures of topics and topics are distributions over '
        'words, estimated via Gibbs sampling or variational Bayes. NMF decomposes the '
        'TF-IDF matrix into topic and document factor matrices. BERTopic clusters BERT '
        'embeddings with HDBSCAN and generates topic labels using c-TF-IDF. Coherence '
        'score Cv is the standard metric for selecting the number of topics.'
    )},
    {'title': 'Question Answering Systems', 'category': 'nlp', 'content': (
        'QA systems answer questions from text. Extractive QA predicts answer spans: '
        'BERT models fine-tuned on SQuAD predict start and end token positions. '
        'Open-domain QA retrieves evidence then extracts answers: DPR (Dense Passage '
        'Retrieval) plus FiD (Fusion-in-Decoder) and RAG (Retrieval Augmented Generation). '
        'Multi-hop QA requires reasoning across multiple documents (HotpotQA, MuSiQue). '
        'Evaluation metrics include EM, F1, ROUGE, and BERTScore.'
    )},
    # Deep Learning
    {'title': 'Convolutional Neural Networks', 'category': 'deep_learning', 'content': (
        'CNNs extract hierarchical spatial features via learnable filters. Convolution '
        'computes output[i,j] = sum of input[i+m, j+n] * kernel[m,n] + bias. '
        'Pooling (max/average) reduces spatial dimensions and adds translation invariance. '
        'Batch normalisation after conv layers stabilises training significantly. '
        'Architectures: VGG (deep, small filters), ResNet (skip connections for gradient '
        'flow), EfficientNet (compound scaling), MobileNet (depthwise separable convolutions).'
    )},
    {'title': 'Recurrent Neural Networks and LSTMs', 'category': 'deep_learning', 'content': (
        'RNNs process sequences by maintaining a hidden state at each time step. '
        'Vanishing gradients prevent learning long-range dependencies. LSTM introduces '
        'forget, input, and output gates that control the memory cell. The forget gate '
        'f_t = sigmoid(W_f [h_{t-1}, x_t] + b_f) decides what to discard. GRU simplifies '
        'with reset and update gates. Bidirectional RNNs process sequences in both '
        'directions. Encoder-decoder with attention led to the transformer revolution.'
    )},
    {'title': 'Generative Adversarial Networks', 'category': 'deep_learning', 'content': (
        'GANs train a Generator G and Discriminator D adversarially: G tries to fool D, '
        'D tries to distinguish real from generated data. Training instabilities include '
        'mode collapse (G produces limited diversity) and vanishing gradients. Improvements: '
        'WGAN uses Wasserstein distance, spectral normalisation, and gradient penalty. '
        'StyleGAN2 produces photorealistic images via style injection. CycleGAN enables '
        'unpaired image-to-image translation. Evaluation uses FID and Inception Score.'
    )},
    {'title': 'Attention Mechanisms', 'category': 'deep_learning', 'content': (
        'Attention allows models to focus on relevant parts of the input. Bahdanau '
        '(additive) attention computes scores as v^T tanh(W*h_t + U*h_s). Scaled '
        'dot-product attention: score = QK^T / sqrt(d_k). Self-attention relates each '
        'position to all others in the same sequence. Multi-head attention projects to h '
        'subspaces, computes attention independently, and concatenates results. Flash '
        'Attention reduces memory from O(N^2) to O(N) using tiling and online softmax.'
    )},
    {'title': 'Neural Architecture Search', 'category': 'deep_learning', 'content': (
        'NAS automates neural architecture design. Search strategies: reinforcement '
        'learning (ENAS), evolutionary algorithms, differentiable NAS (DARTS relaxes '
        'discrete choices to Softmax over operations), one-shot NAS with weight sharing. '
        'Hardware-aware NAS optimises FLOPs, latency, and memory for target devices. '
        'Once-for-All trains a single network supporting many sub-networks. AutoML '
        'broader scope includes HPO (Bayesian optimisation, Optuna) alongside NAS.'
    )},
    # Information Retrieval
    {'title': 'Inverted Index and BM25', 'category': 'ir', 'content': (
        'An inverted index maps terms to posting lists of (doc_id, term_frequency) pairs. '
        'BM25 scoring: sum over query terms t of IDF(t) * tf_normalised. The parameters '
        'k1 (default 1.5) controls TF saturation and b (default 0.75) controls document '
        'length normalisation. IDF(t) = log((N - df(t) + 0.5) / (df(t) + 0.5) + 1). '
        'Compression techniques: variable-byte encoding, Elias-Fano, PFOR-DELTA. '
        'BM25+ extends BM25 with a lower bound on the TF contribution.'
    )},
    {'title': 'Dense Retrieval and DPR', 'category': 'ir', 'content': (
        'Dense retrieval encodes queries and documents as dense vectors and retrieves '
        'by maximum inner product search. DPR uses separate BERT encoders trained with '
        'in-batch negatives and dot-product similarity. Hard negatives from BM25 improve '
        'training quality significantly. ColBERT uses late interaction: max-similarity '
        'score = sum over query tokens of max over passage tokens of (E_q dot E_p). '
        'SPLADE learns sparse representations via BERT MLM head and RELU sparsification. '
        'BEIR benchmark evaluates zero-shot retrieval generalisation across domains.'
    )},
    {'title': 'Reranking Models', 'category': 'ir', 'content': (
        'Rerankers score query-passage pairs with high accuracy. Cross-encoders pass '
        'the concatenated query-passage pair through a transformer for relevance scoring, '
        'which is far more accurate than bi-encoders but requires O(n) forward passes. '
        'MonoBERT and MonoT5 generate true/false tokens and compare their logits. '
        'Listwise rerankers (RankGPT, PRP-Sorting) use LLMs to rank k documents together. '
        'MMR (Maximal Marginal Relevance) balances relevance and diversity in the result set.'
    )},
    {'title': 'Evaluation Metrics for IR', 'category': 'ir', 'content': (
        'Precision@k is the fraction of top-k results that are relevant. Recall@k is the '
        'fraction of relevant documents found in the top-k. Mean Reciprocal Rank (MRR) is '
        'the mean of 1/rank of the first relevant result. Average Precision is the area '
        'under the precision-recall curve. MAP is mean AP across all queries. '
        'nDCG@k accounts for graded relevance and position: DCG/IDCG where '
        'DCG = sum rel_i / log2(i+1). TREC eval is the standard evaluation tool.'
    )},
    {'title': 'Query Expansion and PRF', 'category': 'ir', 'content': (
        'Query expansion adds related terms to improve recall. Pseudo-Relevance Feedback '
        '(PRF) assumes top-k retrieved documents are relevant and extracts expansion terms. '
        'The Rocchio algorithm adds the centroid of top-k documents to the query vector. '
        'RM3 is a strong PRF baseline for BM25 retrieval. Neural query expansion generates '
        'expansion terms with seq2seq models (doc2query, InPars). SPLADE implicitly '
        'expands queries via learned sparse representations.'
    )},
    # Software Engineering
    {'title': 'Design Patterns: Creational', 'category': 'swe', 'content': (
        'Creational patterns abstract object creation. Singleton ensures one instance '
        'per class. Factory Method defines an interface for creation, letting subclasses '
        'decide which class to instantiate. Abstract Factory creates families of related '
        'objects. Builder separates construction from representation using a fluent '
        'interface for objects with many optional parameters. Prototype creates objects '
        'by cloning. In Python, metaclasses enable Singleton and dataclasses enable Builder.'
    )},
    {'title': 'Concurrency and Parallelism', 'category': 'swe', 'content': (
        'Concurrency handles multiple tasks making progress; parallelism runs tasks at the '
        'same instant. The Python GIL limits threading for CPU-bound tasks; multiprocessing '
        'uses separate processes. asyncio provides cooperative multitasking via an event '
        'loop and coroutines (async/await). ThreadPoolExecutor handles IO-bound tasks; '
        'ProcessPoolExecutor handles CPU-bound tasks. Lock, Semaphore, Queue provide '
        'synchronisation. asyncio.gather() runs coroutines concurrently.'
    )},
    {'title': 'API Design Principles', 'category': 'swe', 'content': (
        'Good APIs are easy to use and hard to misuse. REST principles: statelessness, '
        'uniform interface (GET/POST/PUT/DELETE), HATEOAS, layered system. Pagination: '
        'cursor-based (stable under mutations) vs offset-based (simpler). Versioning: '
        'URI (/v1/), header, or query parameter. Rate limiting uses token bucket or sliding '
        'window algorithms. PUT, DELETE, GET are idempotent; POST is not. OpenAPI 3.1 '
        'provides standard documentation. gRPC uses Protobuf for efficient binary transport.'
    )},
    {'title': 'Testing Strategies', 'category': 'swe', 'content': (
        'The testing pyramid: many unit tests, fewer integration tests, fewest E2E tests. '
        'Unit tests are fast, isolated, and mock dependencies. pytest with fixtures, '
        'parametrize, and coverage.py provides comprehensive Python testing. '
        'Property-based testing (Hypothesis) generates inputs automatically. Mutation '
        'testing (mutmut) checks test effectiveness. Contract testing (Pact) verifies '
        'API compatibility across services. TDD uses the Red-Green-Refactor cycle.'
    )},
    {'title': 'System Design: Scalability', 'category': 'swe', 'content': (
        'Horizontal scaling adds more servers; vertical scaling upgrades existing ones. '
        'Stateless services scale horizontally; state requires distributed caches. '
        'Consistent hashing minimises reshuffling when nodes change. CAP theorem: choose '
        'consistency (CP) or availability (AP) under partitions. CQRS separates reads and '
        'writes. Message queues (Kafka, RabbitMQ) decouple producers from consumers. '
        'Backpressure prevents fast producers from overwhelming slow consumers.'
    )},
    # Data Engineering
    {'title': 'Data Pipeline Orchestration', 'category': 'data_eng', 'content': (
        'Pipeline orchestration schedules and monitors data workflows. Airflow uses DAGs of '
        'operators: PythonOperator, BashOperator, and Sensors that wait for conditions. '
        'XCom shares data between tasks; Connections store credentials securely. Prefect '
        'uses Pythonic flow and task decorators with automatic retry and caching. Dagster '
        'provides asset-based workflows with partitions and backfill support. Temporal '
        'handles long-running durable workflows. SLA monitoring alerts on delays.'
    )},
    {'title': 'Streaming Data Processing', 'category': 'data_eng', 'content': (
        'Stream processing ingests and transforms data in real time. Apache Kafka uses '
        'topics with partitions, consumer groups for parallelism, and log retention for '
        'replay. Flink provides exactly-once processing, stateful operators, event time '
        'with watermarks, and tumbling/sliding/session windows. Spark Structured Streaming '
        'supports micro-batch or continuous processing. Kafka Streams is embedded directly '
        'in applications with KTable for stateful aggregations.'
    )},
    {'title': 'Data Lake Architecture', 'category': 'data_eng', 'content': (
        'Data lakes store raw data in its native format at any scale. The Bronze/Silver/Gold '
        'medallion architecture stages data: raw ingest, cleaned/validated, and business-'
        'level aggregations. Parquet and ORC columnar formats provide 10-100x compression '
        'vs CSV with predicate pushdown and column pruning. Delta Lake, Apache Iceberg, and '
        'Apache Hudi add ACID transactions, schema evolution, and time travel. Data '
        'catalogues (Glue, Unity Catalog) track schema and lineage.'
    )},
    {'title': 'Feature Engineering for ML', 'category': 'data_eng', 'content': (
        'Feature engineering transforms raw data into model inputs. Numerical transformations: '
        'standardisation (z-score), min-max scaling, log transform for skewed distributions, '
        'and polynomial features. Categorical: one-hot, ordinal, and target encoding (with '
        'cross-validation to prevent leakage). Temporal: lag features, rolling statistics, '
        'Fourier transforms for seasonality. Feature stores (Feast, Tecton, Hopsworks) serve '
        'precomputed features with consistent offline/online semantics.'
    )},
    {'title': 'Data Quality and Validation', 'category': 'data_eng', 'content': (
        'Data quality dimensions: accuracy, completeness, consistency, timeliness, uniqueness. '
        'Great Expectations defines expectations on columns such as not_null and value_between, '
        'and Checkpoints run them against data. Soda Core provides YAML-based data checks. '
        'dbt tests include unique, not_null, accepted_values, and custom SQL tests. '
        'Anomaly detection on data aggregates uses z-score, IQR, or DBSCAN. Data contracts '
        'formalise producer-consumer agreements for schema and quality.'
    )},
]

def split_article(article, chunk_size=3):
    sentences = re.split(r'(?<=[.!?])\s+', article['content'].strip())
    chunks = []
    step = max(1, chunk_size - 1)
    for i in range(0, len(sentences), step):
        text = ' '.join(sentences[i:i + chunk_size])
        if len(text.strip()) < 20:
            continue
        chunks.append(Document(
            content=text,
            meta={
                'title': article['title'],
                'category': article['category'],
                'article_idx': ARTICLES.index(article),
                'chunk_idx': i // step,
            },
        ))
    return chunks


ALL_CHUNKS = []
for art in ARTICLES:
    ALL_CHUNKS.extend(split_article(art, chunk_size=3))

print(f'Corpus: {len(ARTICLES)} articles -> {len(ALL_CHUNKS)} chunks')
from collections import Counter as _C
cats = _C(c.meta['category'] for c in ALL_CHUNKS)
for cat, n in sorted(cats.items(), key=lambda x: -x[1]):
    print(f'  {cat:<15s}: {n} chunks')
print(f'Avg chunk len: {int(sum(len(c.content) for c in ALL_CHUNKS)/len(ALL_CHUNKS))} chars')

## Shared Embedding Engine

A TF-IDF + random projection engine fitted **once** on the corpus.
All dense retrievers and rerankers share this engine -- same vector space, same semantics.

In [ ]:
class EmbeddingEngine:
    def __init__(self, dim=128):
        self.dim = dim
        self.vocab = {}
        self.idf = {}
        self._proj = None

    def fit(self, texts):
        df = Counter()
        for text in texts:
            for t in set(self._tok(text)):
                df[t] += 1
        self.vocab = {t: i for i, t in enumerate(sorted(df))}
        n = len(texts)
        self.idf = {t: math.log(n / (1 + d)) for t, d in df.items()}
        np.random.seed(42)
        self._proj = np.random.randn(len(self.vocab), self.dim) / math.sqrt(self.dim)

    def _tok(self, text):
        return re.findall(r'[a-z0-9]+', text.lower())

    def _tfidf(self, text):
        tf = Counter(self._tok(text))
        v = np.zeros(len(self.vocab))
        for t, cnt in tf.items():
            if t in self.vocab:
                v[self.vocab[t]] = cnt * self.idf.get(t, 1.0)
        return v

    def encode(self, text):
        v = self._tfidf(text) @ self._proj
        n = np.linalg.norm(v)
        return v / n if n > 0 else v

    def encode_batch(self, texts):
        return np.stack([self.encode(t) for t in texts])

    def similarity(self, a, b):
        return float(np.dot(a, b))


EMB = EmbeddingEngine(dim=128)
EMB.fit([c.content for c in ALL_CHUNKS])

CHUNK_EMBEDDINGS = EMB.encode_batch([c.content for c in ALL_CHUNKS])
for i, doc in enumerate(ALL_CHUNKS):
    doc.embedding = CHUNK_EMBEDDINGS[i]

print(f'EmbeddingEngine: vocab={len(EMB.vocab)}, dim={EMB.dim}')
print(f'Chunk matrix: {CHUNK_EMBEDDINGS.shape}')

## Shared Evaluation Dataset

40 labelled queries across 6 domains and 3 difficulty levels.
Ground truth is at chunk level: a chunk is relevant if it originates from a relevant article.

In [ ]:
EVAL_QUERIES = [
    # ml
    {'query': 'How does Adam optimiser update weights?',
     'relevant_titles': ['Gradient Descent Optimisation'], 'type': 'factual', 'difficulty': 'medium'},
    {'query': 'What is learning rate scheduling?',
     'relevant_titles': ['Gradient Descent Optimisation'], 'type': 'factual', 'difficulty': 'easy'},
    {'query': 'How does L1 regularisation produce sparse weights?',
     'relevant_titles': ['Regularisation Techniques'], 'type': 'conceptual', 'difficulty': 'medium'},
    {'query': 'What is dropout and how does it prevent overfitting?',
     'relevant_titles': ['Regularisation Techniques'], 'type': 'conceptual', 'difficulty': 'easy'},
    {'query': 'Explain multi-head self-attention in transformers',
     'relevant_titles': ['Transformer Architecture', 'Attention Mechanisms'],
     'type': 'conceptual', 'difficulty': 'hard'},
    {'query': 'What is positional encoding in transformers?',
     'relevant_titles': ['Transformer Architecture'], 'type': 'factual', 'difficulty': 'medium'},
    {'query': 'How does k-fold cross-validation work?',
     'relevant_titles': ['Cross-Validation and Model Selection'], 'type': 'procedural', 'difficulty': 'easy'},
    {'query': 'What is data leakage in model evaluation?',
     'relevant_titles': ['Cross-Validation and Model Selection'], 'type': 'conceptual', 'difficulty': 'medium'},
    {'query': 'How does gradient boosting differ from bagging?',
     'relevant_titles': ['Ensemble Methods'], 'type': 'comparison', 'difficulty': 'medium'},
    {'query': 'What is stacking in ensemble learning?',
     'relevant_titles': ['Ensemble Methods'], 'type': 'factual', 'difficulty': 'medium'},
    # nlp
    {'query': 'What is Byte Pair Encoding (BPE)?',
     'relevant_titles': ['Text Tokenisation Strategies'], 'type': 'factual', 'difficulty': 'medium'},
    {'query': 'How does WordPiece differ from BPE tokenisation?',
     'relevant_titles': ['Text Tokenisation Strategies'], 'type': 'comparison', 'difficulty': 'hard'},
    {'query': 'What IOB tagging scheme is used in NER models?',
     'relevant_titles': ['Named Entity Recognition'], 'type': 'factual', 'difficulty': 'medium'},
    {'query': 'How does SBERT generate sentence embeddings?',
     'relevant_titles': ['Sentence Embeddings and Semantic Search'], 'type': 'conceptual', 'difficulty': 'medium'},
    {'query': 'What is HNSW and why is it used for ANN search?',
     'relevant_titles': ['Sentence Embeddings and Semantic Search'], 'type': 'conceptual', 'difficulty': 'hard'},
    {'query': 'How does LDA model topics in documents?',
     'relevant_titles': ['Topic Modelling'], 'type': 'conceptual', 'difficulty': 'medium'},
    {'query': 'How does DPR retrieve passages for open-domain QA?',
     'relevant_titles': ['Question Answering Systems', 'Dense Retrieval and DPR'],
     'type': 'conceptual', 'difficulty': 'hard'},
    # deep learning
    {'query': 'What is the purpose of batch normalisation in CNNs?',
     'relevant_titles': ['Convolutional Neural Networks', 'Regularisation Techniques'],
     'type': 'factual', 'difficulty': 'easy'},
    {'query': 'How do residual skip connections help deep networks train?',
     'relevant_titles': ['Convolutional Neural Networks'], 'type': 'conceptual', 'difficulty': 'medium'},
    {'query': 'What are LSTM gates and what do they control?',
     'relevant_titles': ['Recurrent Neural Networks and LSTMs'], 'type': 'factual', 'difficulty': 'medium'},
    {'query': 'Why does GAN training suffer from mode collapse?',
     'relevant_titles': ['Generative Adversarial Networks'], 'type': 'conceptual', 'difficulty': 'hard'},
    {'query': 'What is FID score used to evaluate in GANs?',
     'relevant_titles': ['Generative Adversarial Networks'], 'type': 'factual', 'difficulty': 'medium'},
    {'query': 'How does Flash Attention reduce memory complexity?',
     'relevant_titles': ['Attention Mechanisms'], 'type': 'conceptual', 'difficulty': 'hard'},
    {'query': 'What is differentiable NAS (DARTS)?',
     'relevant_titles': ['Neural Architecture Search'], 'type': 'factual', 'difficulty': 'hard'},
    # ir
    {'query': 'What are the k1 and b parameters in BM25 scoring?',
     'relevant_titles': ['Inverted Index and BM25'], 'type': 'factual', 'difficulty': 'hard'},
    {'query': 'How does SPLADE combine lexical and semantic matching?',
     'relevant_titles': ['Dense Retrieval and DPR'], 'type': 'conceptual', 'difficulty': 'hard'},
    {'query': 'What is the difference between cross-encoder and bi-encoder?',
     'relevant_titles': ['Reranking Models'], 'type': 'comparison', 'difficulty': 'medium'},
    {'query': 'How does MMR balance relevance and diversity in reranking?',
     'relevant_titles': ['Reranking Models'], 'type': 'conceptual', 'difficulty': 'medium'},
    {'query': 'What metrics are used to evaluate retrieval quality?',
     'relevant_titles': ['Evaluation Metrics for IR'], 'type': 'factual', 'difficulty': 'easy'},
    {'query': 'How does pseudo-relevance feedback expand queries?',
     'relevant_titles': ['Query Expansion and PRF'], 'type': 'conceptual', 'difficulty': 'medium'},
    # swe
    {'query': 'What is the Builder design pattern used for?',
     'relevant_titles': ['Design Patterns: Creational'], 'type': 'factual', 'difficulty': 'easy'},
    {'query': 'How does Python asyncio event loop work?',
     'relevant_titles': ['Concurrency and Parallelism'], 'type': 'conceptual', 'difficulty': 'medium'},
    {'query': 'What is idempotency in REST API design?',
     'relevant_titles': ['API Design Principles'], 'type': 'factual', 'difficulty': 'easy'},
    {'query': 'What is property-based testing and when is it used?',
     'relevant_titles': ['Testing Strategies'], 'type': 'factual', 'difficulty': 'easy'},
    {'query': 'How does consistent hashing help with horizontal scaling?',
     'relevant_titles': ['System Design: Scalability'], 'type': 'conceptual', 'difficulty': 'hard'},
    # data_eng
    {'query': 'How does Apache Airflow schedule DAG tasks?',
     'relevant_titles': ['Data Pipeline Orchestration'], 'type': 'factual', 'difficulty': 'easy'},
    {'query': 'What is exactly-once processing in Apache Flink?',
     'relevant_titles': ['Streaming Data Processing'], 'type': 'factual', 'difficulty': 'medium'},
    {'query': 'What is the medallion architecture in data lakes?',
     'relevant_titles': ['Data Lake Architecture'], 'type': 'factual', 'difficulty': 'easy'},
    {'query': 'What is target encoding and why can it cause leakage?',
     'relevant_titles': ['Feature Engineering for ML'], 'type': 'conceptual', 'difficulty': 'medium'},
    {'query': 'How does Great Expectations validate data quality?',
     'relevant_titles': ['Data Quality and Validation'], 'type': 'factual', 'difficulty': 'easy'},
]

title_to_chunk_ids = defaultdict(set)
for doc in ALL_CHUNKS:
    title_to_chunk_ids[doc.meta['title']].add(doc.id)

for q in EVAL_QUERIES:
    q['relevant_ids'] = set()
    for title in q['relevant_titles']:
        q['relevant_ids'].update(title_to_chunk_ids.get(title, set()))

matched = sum(1 for q in EVAL_QUERIES if q['relevant_ids'])
print(f'Eval queries: {len(EVAL_QUERIES)} ({matched} with ground truth)')
print(f'Types: {dict(Counter(q["type"] for q in EVAL_QUERIES))}')
print(f'Difficulty: {dict(Counter(q["difficulty"] for q in EVAL_QUERIES))}')

## Evaluation Metrics

Implemented once, called by every retriever and reranker combination.

In [ ]:
def recall_at_k(ids, relevant, k):
    if not relevant: return 0.0
    return len(set(ids[:k]) & relevant) / len(relevant)

def precision_at_k(ids, relevant, k):
    if not ids[:k]: return 0.0
    return sum(1 for i in ids[:k] if i in relevant) / k

def mrr(ids, relevant):
    for rank, i in enumerate(ids, 1):
        if i in relevant: return 1.0 / rank
    return 0.0

def ndcg_at_k(ids, relevant, k):
    def dcg(lst):
        return sum((1.0 if i in relevant else 0.0) / math.log2(r + 2)
                   for r, i in enumerate(lst[:k]))
    ideal = sorted(ids, key=lambda i: i in relevant, reverse=True)
    idcg = dcg(ideal)
    return dcg(ids) / idcg if idcg > 0 else 0.0

def success_at_k(ids, relevant, k):
    return 1.0 if set(ids[:k]) & relevant else 0.0

def evaluate(retrieved_ids, relevant, k=10):
    return {
        f'recall@{k}':    recall_at_k(retrieved_ids, relevant, k),
        f'precision@{k}': precision_at_k(retrieved_ids, relevant, k),
        'mrr':            mrr(retrieved_ids, relevant),
        f'ndcg@{k}':      ndcg_at_k(retrieved_ids, relevant, k),
        f'success@{k}':   success_at_k(retrieved_ids, relevant, k),
    }


print('Metrics: Recall@K, Precision@K, MRR, nDCG@K, Success@K')
test_ids = ['a', 'b', 'c', 'd', 'e']
test_rel = {'b', 'd'}
out = evaluate(test_ids, test_rel, k=5)
for m, v in out.items():
    print(f'  {m}: {v:.4f}')

## Six Retrieval Strategies

Each retriever exposes one interface: `retrieve(query, top_k) -> (list[Document], latency_ms)`.

### Strategy 1 -- BM25 (Sparse Lexical)

The de-facto baseline for keyword retrieval. Fast, interpretable, no training needed.

In [ ]:
class BM25Retriever:
    name = 'BM25'

    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs = docs
        self.k1, self.b = k1, b
        self._tok_map = {}
        self._df = Counter()
        for d in docs:
            toks = self._tok(d.content)
            self._tok_map[d.id] = toks
            for t in set(toks):
                self._df[t] += 1
        lens = [len(t) for t in self._tok_map.values()]
        self._avgdl = float(np.mean(lens)) if lens else 1.0
        self._n = len(docs)

    def _tok(self, text):
        return re.findall(r'[a-z0-9]+', text.lower())

    def retrieve(self, query, top_k=10):
        t0 = time.perf_counter()
        qtoks = self._tok(query)
        scores = {}
        for doc in self.docs:
            toks = self._tok_map[doc.id]
            tf_map = Counter(toks)
            dl = len(toks)
            s = 0.0
            for qt in qtoks:
                tf = tf_map.get(qt, 0)
                df = self._df.get(qt, 0)
                idf = math.log((self._n - df + 0.5) / (df + 0.5) + 1)
                tf_n = tf * (self.k1 + 1) / (
                    tf + self.k1 * (1 - self.b + self.b * dl / self._avgdl))
                s += idf * tf_n
            if s > 0:
                scores[doc.id] = s
        ranked = sorted(scores.items(), key=lambda x: -x[1])[:top_k]
        id_to_doc = {d.id: d for d in self.docs}
        results = []
        for did, sc in ranked:
            d = copy.copy(id_to_doc[did])
            d.score = round(sc, 5)
            results.append(d)
        return results, (time.perf_counter() - t0) * 1000


print('BM25Retriever defined.')

### Strategy 2 -- TF-IDF Cosine (Sparse Vector)

In [ ]:
class TFIDFRetriever:
    name = 'TF-IDF Cosine'

    def __init__(self, docs):
        self.docs = docs
        self._build()

    def _tok(self, text):
        return re.findall(r'[a-z0-9]+', text.lower())

    def _build(self):
        df = Counter()
        for d in self.docs:
            for t in set(self._tok(d.content)):
                df[t] += 1
        n = len(self.docs)
        self._idf = {t: math.log(n / (1 + v)) for t, v in df.items()}
        self._vocab = {t: i for i, t in enumerate(sorted(self._idf))}
        mat = np.zeros((len(self.docs), len(self._vocab)))
        for i, d in enumerate(self.docs):
            toks = Counter(self._tok(d.content))
            for t, cnt in toks.items():
                if t in self._vocab:
                    mat[i, self._vocab[t]] = cnt * self._idf.get(t, 0)
            norm = np.linalg.norm(mat[i])
            if norm > 0:
                mat[i] /= norm
        self._mat = mat

    def _qvec(self, query):
        toks = Counter(self._tok(query))
        v = np.zeros(len(self._vocab))
        for t, cnt in toks.items():
            if t in self._vocab:
                v[self._vocab[t]] = cnt * self._idf.get(t, 0)
        norm = np.linalg.norm(v)
        return v / norm if norm > 0 else v

    def retrieve(self, query, top_k=10):
        t0 = time.perf_counter()
        qv = self._qvec(query)
        sims = self._mat @ qv
        topk = np.argsort(sims)[::-1][:top_k]
        results = []
        for idx in topk:
            if sims[idx] <= 0: continue
            d = copy.copy(self.docs[idx])
            d.score = round(float(sims[idx]), 5)
            results.append(d)
        return results, (time.perf_counter() - t0) * 1000


print('TFIDFRetriever defined.')

### Strategy 3 -- Dense Embedding (Semantic)

In [ ]:
class DenseRetriever:
    name = 'Dense Embedding'

    def __init__(self, docs, engine):
        self.docs = docs
        self.engine = engine
        self._emb_mat = np.stack([d.embedding for d in self.docs])

    def retrieve(self, query, top_k=10):
        t0 = time.perf_counter()
        qe = self.engine.encode(query)
        sims = self._emb_mat @ qe
        topk = np.argsort(sims)[::-1][:top_k]
        results = []
        for idx in topk:
            d = copy.copy(self.docs[idx])
            d.score = round(float(sims[idx]), 5)
            results.append(d)
        return results, (time.perf_counter() - t0) * 1000


print('DenseRetriever defined.')

### Strategy 4 -- ColBERT-style MaxSim (Late Interaction)

ColBERT encodes each **token** of the query and document separately,
then scores relevance as the sum of maximum per-token similarities.

```
Score(q, d) = sum over q_tokens of max over d_tokens of cosine(E_q, E_d)
```

In production: `pylate` library or `Ragatouille`.

In [ ]:
class ColBERTRetriever:
    name = 'ColBERT MaxSim'

    def __init__(self, docs, engine):
        self.docs = docs
        self.engine = engine
        self._doc_tok_embs = []
        for d in self.docs:
            tokens = re.findall(r'[a-z0-9]+', d.content.lower())
            if tokens:
                embs = np.stack([engine.encode(t) for t in tokens[:30]])
            else:
                embs = np.zeros((1, engine.dim))
            self._doc_tok_embs.append(embs)

    def _maxsim(self, q_embs, d_embs):
        sims = q_embs @ d_embs.T
        return float(sims.max(axis=1).sum())

    def retrieve(self, query, top_k=10):
        t0 = time.perf_counter()
        qtoks = re.findall(r'[a-z0-9]+', query.lower())
        if not qtoks: return [], 0.0
        q_embs = np.stack([self.engine.encode(t) for t in qtoks])
        scores = [self._maxsim(q_embs, de) for de in self._doc_tok_embs]
        topk = np.argsort(scores)[::-1][:top_k]
        results = []
        for idx in topk:
            d = copy.copy(self.docs[idx])
            d.score = round(scores[idx], 5)
            results.append(d)
        return results, (time.perf_counter() - t0) * 1000


print('ColBERTRetriever defined.')

### Strategies 5 & 6 -- Hybrid Retrievers

Both fuse BM25 (sparse) and Dense (semantic) results.

- **RRF (Reciprocal Rank Fusion)**: rank-based, robust to score distribution differences
- **Linear**: normalises and weights the raw scores directly

In [ ]:
class HybridRRFRetriever:
    name = 'Hybrid RRF'

    def __init__(self, bm25, dense, k_rrf=60, fetch_k=25):
        self.bm25 = bm25
        self.dense = dense
        self.k_rrf = k_rrf
        self.fetch_k = fetch_k

    def retrieve(self, query, top_k=10):
        t0 = time.perf_counter()
        bm25_docs, _ = self.bm25.retrieve(query, top_k=self.fetch_k)
        dense_docs, _ = self.dense.retrieve(query, top_k=self.fetch_k)
        rrf = {}
        doc_map = {}
        for rank, d in enumerate(bm25_docs):
            rrf[d.id] = rrf.get(d.id, 0) + 1.0 / (self.k_rrf + rank + 1)
            doc_map[d.id] = d
        for rank, d in enumerate(dense_docs):
            rrf[d.id] = rrf.get(d.id, 0) + 1.0 / (self.k_rrf + rank + 1)
            if d.id not in doc_map:
                doc_map[d.id] = d
        ranked = sorted(rrf.items(), key=lambda x: -x[1])[:top_k]
        results = []
        for did, sc in ranked:
            d = copy.copy(doc_map[did])
            d.score = round(sc, 6)
            results.append(d)
        return results, (time.perf_counter() - t0) * 1000


class HybridLinearRetriever:
    name = 'Hybrid Linear'

    def __init__(self, bm25, dense, alpha=0.5, fetch_k=25):
        self.bm25 = bm25
        self.dense = dense
        self.alpha = alpha
        self.fetch_k = fetch_k

    @staticmethod
    def _norm(docs):
        if not docs: return {}
        scores = [d.score or 0 for d in docs]
        mn, mx = min(scores), max(scores)
        denom = mx - mn if mx > mn else 1.0
        return {d.id: (d.score - mn) / denom for d in docs}

    def retrieve(self, query, top_k=10):
        t0 = time.perf_counter()
        bm25_docs, _ = self.bm25.retrieve(query, top_k=self.fetch_k)
        dense_docs, _ = self.dense.retrieve(query, top_k=self.fetch_k)
        bn = self._norm(bm25_docs)
        dn = self._norm(dense_docs)
        doc_map = {d.id: d for d in bm25_docs + dense_docs}
        combined = {did: (1 - self.alpha) * bn.get(did, 0) + self.alpha * dn.get(did, 0)
                    for did in doc_map}
        ranked = sorted(combined.items(), key=lambda x: -x[1])[:top_k]
        results = []
        for did, sc in ranked:
            d = copy.copy(doc_map[did])
            d.score = round(sc, 5)
            results.append(d)
        return results, (time.perf_counter() - t0) * 1000


print('Hybrid retrievers defined.')

## Instantiate All Retrievers

In [ ]:
bm25_ret    = BM25Retriever(ALL_CHUNKS)
tfidf_ret   = TFIDFRetriever(ALL_CHUNKS)
dense_ret   = DenseRetriever(ALL_CHUNKS, EMB)
colbert_ret = ColBERTRetriever(ALL_CHUNKS, EMB)
rrf_ret     = HybridRRFRetriever(bm25_ret, dense_ret, fetch_k=25)
linear_ret  = HybridLinearRetriever(bm25_ret, dense_ret, alpha=0.5, fetch_k=25)

RETRIEVERS = [bm25_ret, tfidf_ret, dense_ret, colbert_ret, rrf_ret, linear_ret]

print('Retrievers:')
for r in RETRIEVERS:
    print(f'  {r.name}')

demo_q = 'How does attention work in transformers?'
for r in RETRIEVERS[:3]:
    docs, ms = r.retrieve(demo_q, top_k=3)
    title = docs[0].meta['title'] if docs else 'none'
    print(f'  [{r.name}] top: {title} ({ms:.1f}ms)')

## Four Reranking Strategies

Each reranker receives top-N first-stage candidates and returns a reordered list.

### Reranker 1 -- Cross-Encoder (Pointwise)

Concatenates (query, document) and scores via full cross-attention.
Most accurate but $O(n)$ forward passes per query.

**Haystack production:** `TransformersSimilarityRanker(model='cross-encoder/ms-marco-MiniLM-L-6-v2')`  
**LangChain production:** `CrossEncoderReranker(model=HuggingFaceCrossEncoder(...))`

In [ ]:
class CrossEncoderReranker:
    name = 'Cross-Encoder'

    def __init__(self, engine):
        self.engine = engine

    def _score(self, query, doc_text):
        qtoks = re.findall(r'[a-z0-9]+', query.lower())
        dtoks = re.findall(r'[a-z0-9]+', doc_text.lower())
        if not qtoks or not dtoks: return 0.0
        qe = np.stack([self.engine.encode(t) for t in qtoks[:10]])
        de = np.stack([self.engine.encode(t) for t in dtoks[:20]])
        sim_mat = qe @ de.T
        row_max = sim_mat.max(axis=1).mean()
        col_max = sim_mat.max(axis=0).mean()
        raw = (row_max + col_max) / 2.0
        return float(raw + np.random.normal(0, 0.03))

    def rerank(self, query, candidates, top_k=5):
        t0 = time.perf_counter()
        scored = sorted([(self._score(query, d.content), d)
                         for d in candidates], key=lambda x: -x[0])
        results = []
        for sc, d in scored[:top_k]:
            d2 = copy.copy(d)
            d2.score = round(sc, 5)
            results.append(d2)
        return results, (time.perf_counter() - t0) * 1000


print('CrossEncoderReranker defined.')

### Reranker 2 -- MonoT5-style (Generative)

Frames relevance as text generation: score = logit('true') - logit('false').
Prompt template: `Query: {q} Document: {d} Relevant:`

**Production:** `sentence-transformers/ms-marco-MiniLM-L-12-v2` or `castorini/monot5-base-msmarco`

In [ ]:
class MonoT5Reranker:
    name = 'MonoT5-style'

    def __init__(self, engine):
        self.engine = engine

    def _logit(self, query, doc_text):
        qe = self.engine.encode(query)
        de = self.engine.encode(doc_text)
        dense = float(np.dot(qe, de))
        qtoks = set(re.findall(r'[a-z0-9]+', query.lower()))
        dtoks = set(re.findall(r'[a-z0-9]+', doc_text.lower()))
        overlap = len(qtoks & dtoks) / max(len(qtoks), 1)
        return dense + 0.3 * overlap + np.random.normal(0, 0.02)

    def rerank(self, query, candidates, top_k=5):
        t0 = time.perf_counter()
        scored = sorted([(self._logit(query, d.content), d)
                         for d in candidates], key=lambda x: -x[0])
        results = []
        for sc, d in scored[:top_k]:
            d2 = copy.copy(d)
            d2.score = round(sc, 5)
            results.append(d2)
        return results, (time.perf_counter() - t0) * 1000


print('MonoT5Reranker defined.')

### Reranker 3 -- Cohere-style API Reranker

Simulates hosted API rerankers (Cohere Rerank, Jina Reranker) with a richer scoring
function and realistic API latency.

**LangChain production:**
```python
from langchain.retrievers.document_compressors import CohereRerank
compressor = CohereRerank(cohere_api_key='...', top_n=5)
```

In [ ]:
class CohereStyleReranker:
    name = 'Cohere-style API'

    def __init__(self, engine, api_latency_ms=40.0):
        self.engine = engine
        self._api_latency = api_latency_ms / 1000

    def _api_score(self, query, doc_text):
        qe = self.engine.encode(query)
        de = self.engine.encode(doc_text)
        dense = float(np.dot(qe, de))
        qtoks = Counter(re.findall(r'[a-z0-9]+', query.lower()))
        dtoks = Counter(re.findall(r'[a-z0-9]+', doc_text.lower()))
        shared = sum(min(qtoks[t], dtoks[t]) for t in qtoks if t in dtoks)
        tf_sim = shared / max(sum(qtoks.values()), 1)
        length_bonus = min(0.08, len(doc_text) / 3000)
        return dense + 0.4 * tf_sim + length_bonus + np.random.normal(0, 0.02)

    def rerank(self, query, candidates, top_k=5):
        t0 = time.perf_counter()
        time.sleep(self._api_latency * len(candidates) / 20)
        scored = sorted([(self._api_score(query, d.content), d)
                         for d in candidates], key=lambda x: -x[0])
        results = []
        for sc, d in scored[:top_k]:
            d2 = copy.copy(d)
            d2.score = round(sc, 5)
            results.append(d2)
        return results, (time.perf_counter() - t0) * 1000


print('CohereStyleReranker defined.')

### Reranker 4 -- MMR (Maximal Marginal Relevance)

Explicitly trades off relevance vs diversity. At each step selects the document
that is most relevant to the query **and** least similar to already-selected results.

$$\text{MMR}(d) = \lambda \cdot \text{Sim}(d, q) - (1-\lambda) \cdot \max_{j \in S} \text{Sim}(d, d_j)$$

**LangChain production:** `vectorstore.max_marginal_relevance_search(query, k=5, lambda_mult=0.5)`

In [ ]:
class MMRReranker:
    name = 'MMR Diversity'

    def __init__(self, engine, lambda_mult=0.6):
        self.engine = engine
        self.lam = lambda_mult

    def rerank(self, query, candidates, top_k=5):
        t0 = time.perf_counter()
        if not candidates: return [], 0.0
        qe = self.engine.encode(query)
        doc_embs = [self.engine.encode(d.content) for d in candidates]
        q_sims = [float(np.dot(qe, de)) for de in doc_embs]
        selected = []
        remaining = list(range(len(candidates)))
        while len(selected) < top_k and remaining:
            if not selected:
                best = max(remaining, key=lambda i: q_sims[i])
            else:
                def mmr_score(i):
                    rel = self.lam * q_sims[i]
                    redundancy = (1 - self.lam) * max(
                        float(np.dot(doc_embs[i], doc_embs[j])) for j in selected)
                    return rel - redundancy
                best = max(remaining, key=mmr_score)
            selected.append(best)
            remaining.remove(best)
        results = []
        for idx in selected:
            d = copy.copy(candidates[idx])
            d.score = round(q_sims[idx], 5)
            results.append(d)
        return results, (time.perf_counter() - t0) * 1000


print('MMRReranker defined.')

In [ ]:
cross_enc = CrossEncoderReranker(EMB)
mono_t5   = MonoT5Reranker(EMB)
cohere_rr = CohereStyleReranker(EMB, api_latency_ms=35.0)
mmr_rr    = MMRReranker(EMB, lambda_mult=0.6)

RERANKERS = [cross_enc, mono_t5, cohere_rr, mmr_rr]

print('Rerankers:')
for r in RERANKERS:
    print(f'  {r.name}')

## Full Evaluation

### Retriever-Only Baseline

Evaluate all 6 retrievers **without reranking** at K=10.

In [ ]:
K_RETRIEVE = 20
K_EVAL     = 10

retriever_results = []
for ret in RETRIEVERS:
    for eq in EVAL_QUERIES:
        docs, lat = ret.retrieve(eq['query'], top_k=K_RETRIEVE)
        ids = [d.id for d in docs]
        m = evaluate(ids, eq['relevant_ids'], k=K_EVAL)
        retriever_results.append({
            'retriever': ret.name,
            'query': eq['query'][:55],
            'query_type': eq['type'],
            'difficulty': eq['difficulty'],
            'latency_ms': lat,
            **m,
        })

ret_df = pd.DataFrame(retriever_results)

REC_COL  = f'recall@{K_EVAL}'
PREC_COL = f'precision@{K_EVAL}'
NDCG_COL = f'ndcg@{K_EVAL}'
SUC_COL  = f'success@{K_EVAL}'

ret_summary = ret_df.groupby('retriever').agg(
    recall   =(REC_COL,  'mean'),
    precision=(PREC_COL, 'mean'),
    mrr      =('mrr',    'mean'),
    ndcg     =(NDCG_COL, 'mean'),
    success  =(SUC_COL,  'mean'),
    latency  =('latency_ms', 'mean'),
).round(4)

print('RETRIEVER BASELINE @K=10')
print('=' * 70)
print(ret_summary.to_string())

### Retriever + Reranker Combinations

Apply each reranker to the top-3 retrievers and evaluate @K=5.
This tests whether reranking improves precision beyond the first-stage results.

In [ ]:
K_RERANK_IN  = 15
K_RERANK_OUT = 5

top_retrievers = [bm25_ret, dense_ret, rrf_ret]

reranker_results = []
for ret in top_retrievers:
    for rr in RERANKERS:
        for eq in EVAL_QUERIES:
            cands, ret_lat = ret.retrieve(eq['query'], top_k=K_RERANK_IN)
            reranked, rr_lat = rr.rerank(eq['query'], cands, top_k=K_RERANK_OUT)
            ids = [d.id for d in reranked]
            m = evaluate(ids, eq['relevant_ids'], k=K_RERANK_OUT)
            reranker_results.append({
                'retriever': ret.name,
                'reranker': rr.name,
                'combo': f'{ret.name} + {rr.name}',
                'query': eq['query'][:55],
                'query_type': eq['type'],
                'difficulty': eq['difficulty'],
                'ret_latency_ms': ret_lat,
                'rr_latency_ms': rr_lat,
                'total_latency_ms': ret_lat + rr_lat,
                **m,
            })

rr_df = pd.DataFrame(reranker_results)

P_COL5 = f'precision@{K_RERANK_OUT}'
N_COL5 = f'ndcg@{K_RERANK_OUT}'

rr_summary = rr_df.groupby(['retriever', 'reranker']).agg(
    precision=(P_COL5, 'mean'),
    mrr      =('mrr',  'mean'),
    ndcg     =(N_COL5, 'mean'),
    total_ms =('total_latency_ms', 'mean'),
).round(4)

print(f'RETRIEVER + RERANKER @K={K_RERANK_OUT}')
print('=' * 70)
print(rr_summary.to_string())

## Visualisations

### 1. Retriever Baseline Comparison

In [ ]:
COLORS = {
    'BM25':            '#3498db',
    'TF-IDF Cosine':   '#9b59b6',
    'Dense Embedding': '#e74c3c',
    'ColBERT MaxSim':  '#f39c12',
    'Hybrid RRF':      '#2ecc71',
    'Hybrid Linear':   '#1abc9c',
}

metrics_to_plot = [
    ('recall',    'Recall@10'),
    ('mrr',       'MRR'),
    ('ndcg',      'nDCG@10'),
    ('success',   'Success@10'),
]

fig = make_subplots(rows=1, cols=4,
                    subplot_titles=[m[1] for m in metrics_to_plot])

for col, (metric, label) in enumerate(metrics_to_plot, 1):
    order = ret_summary[metric].sort_values(ascending=False)
    for ret_name in order.index:
        fig.add_trace(go.Bar(
            x=[ret_name], y=[order[ret_name]],
            marker_color=COLORS.get(ret_name, '#7f8c8d'),
            name=ret_name, showlegend=(col == 1),
        ), row=1, col=col)

fig.update_layout(height=430, template='plotly_white',
                  title_text='Retriever Baseline Comparison (no reranking)',
                  barmode='group')
fig.update_yaxes(range=[0, 1])
fig.show()

### 2. Precision@5 Lift from Reranking

In [ ]:
# Retriever-only at K=5 for fair comparison
ret5_rows = []
for ret in top_retrievers:
    for eq in EVAL_QUERIES:
        docs, lat = ret.retrieve(eq['query'], top_k=K_RERANK_OUT)
        ids = [d.id for d in docs]
        m = evaluate(ids, eq['relevant_ids'], k=K_RERANK_OUT)
        ret5_rows.append({
            'combo': f'{ret.name} (no rerank)',
            P_COL5: m[P_COL5], 'mrr': m['mrr'],
        })

ret5_df = pd.DataFrame(ret5_rows)

plot_df = pd.concat([
    ret5_df.groupby('combo')[[P_COL5, 'mrr']].mean().reset_index(),
    rr_df.groupby('combo')[[P_COL5, 'mrr']].mean().reset_index(),
]).sort_values(P_COL5, ascending=False)

fig = px.bar(plot_df.head(12), x='combo', y=P_COL5, color='combo',
             title=f'Precision@{K_RERANK_OUT}: Retriever-only vs Retriever+Reranker',
             template='plotly_white',
             labels={'combo': ''})
fig.update_layout(height=430, showlegend=False, xaxis_tickangle=-30)
fig.update_yaxes(range=[0, 1])
fig.show()

### 3. Quality--Latency Trade-off

In [ ]:
scatter_rows = []
for ret in RETRIEVERS:
    lats, mrrs = [], []
    for eq in EVAL_QUERIES:
        docs, lat = ret.retrieve(eq['query'], top_k=K_EVAL)
        ids = [d.id for d in docs]
        m = evaluate(ids, eq['relevant_ids'], k=K_EVAL)
        lats.append(lat); mrrs.append(m['mrr'])
    scatter_rows.append({
        'name': ret.name, 'type': 'retriever-only',
        'latency_ms': float(np.mean(lats)),
        'mrr': float(np.mean(mrrs)),
    })

for (ret_name, rr_name), grp in rr_df.groupby(['retriever', 'reranker']):
    scatter_rows.append({
        'name': f'{ret_name[:5]}+{rr_name[:5]}',
        'type': 'retriever+reranker',
        'latency_ms': float(grp['total_latency_ms'].mean()),
        'mrr': float(grp['mrr'].mean()),
    })

scatter_df = pd.DataFrame(scatter_rows)

fig = px.scatter(
    scatter_df, x='latency_ms', y='mrr',
    color='type', text='name',
    title='Quality (MRR) vs Latency Trade-off',
    labels={'latency_ms': 'Mean Latency (ms)', 'mrr': 'Mean MRR'},
    template='plotly_white',
    color_discrete_map={'retriever-only': '#3498db', 'retriever+reranker': '#e74c3c'},
)
fig.update_traces(textposition='top center', marker_size=10)
fig.update_layout(height=500)
fig.show()

### 4. Reranker MRR Lift Heatmap

In [ ]:
ret5_mrr = {}
for ret in top_retrievers:
    mrrs = []
    for eq in EVAL_QUERIES:
        docs, _ = ret.retrieve(eq['query'], top_k=K_RERANK_OUT)
        ids = [d.id for d in docs]
        mrrs.append(mrr(ids, eq['relevant_ids']))
    ret5_mrr[ret.name] = float(np.mean(mrrs))

lift_data = []
for (ret_name, rr_name), grp in rr_df.groupby(['retriever', 'reranker']):
    baseline = ret5_mrr.get(ret_name, 0)
    lift_data.append({
        'Retriever': ret_name, 'Reranker': rr_name,
        'MRR Lift': round(float(grp['mrr'].mean()) - baseline, 4)
    })

lift_df = pd.DataFrame(lift_data).pivot(
    index='Retriever', columns='Reranker', values='MRR Lift'
)

fig = px.imshow(
    lift_df.values,
    x=lift_df.columns.tolist(),
    y=lift_df.index.tolist(),
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    text_auto='.4f',
    title='MRR Lift from Reranking (vs retriever-only @K=5)',
    labels=dict(x='Reranker', y='Retriever', color='MRR Lift'),
)
fig.update_layout(template='plotly_white', height=340)
fig.show()

### 5. Performance by Query Type and Difficulty

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

type_ret = ret_df.groupby(['retriever', 'query_type'])['mrr'].mean().unstack('retriever')
type_ret.plot(kind='bar', ax=axes[0], colormap='tab10',
              rot=0, edgecolor='white', linewidth=0.5)
axes[0].set_title('MRR by Query Type (retriever-only)')
axes[0].set_xlabel('Query Type')
axes[0].set_ylabel('Mean MRR')
axes[0].set_ylim(0, 1)
axes[0].legend(fontsize=7, loc='upper right')

diff_ret = ret_df.groupby(['retriever', 'difficulty'])['mrr'].mean().unstack('retriever')
diff_ret = diff_ret.reindex(['easy', 'medium', 'hard'])
diff_ret.plot(kind='bar', ax=axes[1], colormap='tab10',
              rot=0, edgecolor='white', linewidth=0.5)
axes[1].set_title('MRR by Difficulty (retriever-only)')
axes[1].set_xlabel('Difficulty')
axes[1].set_ylabel('Mean MRR')
axes[1].set_ylim(0, 1)
axes[1].legend(fontsize=7, loc='upper right')

plt.tight_layout()
plt.show()

### 6. Retriever Profile Radar Chart

In [ ]:
radar_cols = ['recall', 'precision', 'mrr', 'ndcg', 'success']
radar_labels = ['Recall@10', 'Prec@10', 'MRR', 'nDCG@10', 'Success@10']

fig = go.Figure()
colors_list = list(COLORS.values())
for i, ret_name in enumerate(ret_summary.index):
    vals = [float(ret_summary.loc[ret_name, c]) for c in radar_cols]
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=radar_labels + [radar_labels[0]],
        name=ret_name, fill='toself',
        line_color=colors_list[i % len(colors_list)],
        opacity=0.65,
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(range=[0, 1], visible=True)),
    title='Retriever Quality Radar',
    template='plotly_white',
    height=500,
)
fig.show()

### 7. Recall@K Curves (Retrieval Depth)

In [ ]:
k_vals = [1, 3, 5, 10, 15, 20]

fig = go.Figure()
for ret in RETRIEVERS:
    curve = []
    for k in k_vals:
        recalls = []
        for eq in EVAL_QUERIES:
            docs, _ = ret.retrieve(eq['query'], top_k=k)
            ids = [d.id for d in docs]
            recalls.append(recall_at_k(ids, eq['relevant_ids'], k=k))
        curve.append(float(np.mean(recalls)))
    fig.add_trace(go.Scatter(
        x=k_vals, y=curve, mode='lines+markers',
        name=ret.name,
        line=dict(color=COLORS.get(ret.name, '#7f8c8d'), width=2),
        marker=dict(size=7),
    ))

fig.update_layout(
    title='Recall@K Curves -- higher K means broader retrieval',
    xaxis_title='K', yaxis_title='Mean Recall@K',
    template='plotly_white', height=400, yaxis_range=[0, 1],
)
fig.show()

## Deep Dives

### Where Each Retriever Wins

Queries where retrievers disagree most reveal each strategy's strengths.

In [ ]:
ret_names_6 = [r.name for r in RETRIEVERS]

pivot_mrr = ret_df.pivot_table(
    index='query', columns='retriever', values='mrr'
)
pivot_mrr['best']   = pivot_mrr[ret_names_6].idxmax(axis=1)
pivot_mrr['spread'] = pivot_mrr[ret_names_6].apply(
    lambda r: r.max() - r.min(), axis=1
)

top_disagree = pivot_mrr.nlargest(8, 'spread')

print('QUERIES WITH HIGHEST RETRIEVER DISAGREEMENT')
print('=' * 72)
for query, row in top_disagree.iterrows():
    print(f'  Q: {query}')
    scores = {r: row[r] for r in ret_names_6 if r in row and not pd.isna(row[r])}
    for rname, sc in sorted(scores.items(), key=lambda x: -x[1]):
        marker = ' <-- BEST' if rname == row['best'] else ''
        print(f'    {rname:<22s}: {sc:.3f}{marker}')
    print()

print('HOW OFTEN EACH RETRIEVER IS BEST:')
best_counts = pivot_mrr['best'].value_counts()
for name, count in best_counts.items():
    print(f'  {name:<22s}: {count:>3}  {"#" * count}')

### Does Reranking Always Help?

The two-stage pipeline can **hurt** precision if the first-stage already ranked well.
We count queries where reranking improved, maintained, or hurt MRR.

In [ ]:
by_rr = defaultdict(list)

for eq in EVAL_QUERIES:
    cands_base, _ = rrf_ret.retrieve(eq['query'], top_k=K_RERANK_OUT)
    ids_base = [d.id for d in cands_base]
    m1 = mrr(ids_base, eq['relevant_ids'])

    for rr in RERANKERS:
        cands, _ = rrf_ret.retrieve(eq['query'], top_k=K_RERANK_IN)
        reranked, _ = rr.rerank(eq['query'], cands, top_k=K_RERANK_OUT)
        ids2 = [d.id for d in reranked]
        m2 = mrr(ids2, eq['relevant_ids'])
        by_rr[rr.name].append(m2 - m1)

rr_impact = pd.DataFrame({
    rr_name: {
        'mean_delta': round(float(np.mean(deltas)), 4),
        'pct_improved': round(float(np.mean([d > 0.01 for d in deltas])) * 100, 1),
        'pct_hurt':    round(float(np.mean([d < -0.01 for d in deltas])) * 100, 1),
    }
    for rr_name, deltas in by_rr.items()
}).T

print('RERANKER IMPACT ON MRR (base: Hybrid RRF @K=5)')
print(rr_impact.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(RERANKERS))
w = 0.25

ax.bar(x - w, [rr_impact.loc[r.name, 'pct_improved'] for r in RERANKERS],
       width=w, label='% Improved', color='#2ecc71', alpha=0.85)
ax.bar(x,     [100 - rr_impact.loc[r.name, 'pct_improved']
               - rr_impact.loc[r.name, 'pct_hurt'] for r in RERANKERS],
       width=w, label='% Same', color='#95a5a6', alpha=0.85)
ax.bar(x + w, [rr_impact.loc[r.name, 'pct_hurt'] for r in RERANKERS],
       width=w, label='% Hurt', color='#e74c3c', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([r.name for r in RERANKERS])
ax.set_ylabel('% of Queries')
ax.set_title('Reranker Impact vs Hybrid RRF Baseline')
ax.legend()
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

### Side-by-Side Example: All Strategies on One Query

In [ ]:
demo_q = 'What is the difference between cross-encoder and bi-encoder?'
demo_rel = set()
for eq in EVAL_QUERIES:
    if 'cross-encoder' in eq['query'].lower() or 'bi-encoder' in eq['query'].lower():
        demo_rel.update(eq['relevant_ids'])
if not demo_rel:
    demo_rel = EVAL_QUERIES[26]['relevant_ids']

print(f'Query: {demo_q}')
print(f'Relevant chunks: {len(demo_rel)}')
print('=' * 70)

for ret in RETRIEVERS:
    docs, lat = ret.retrieve(demo_q, top_k=5)
    ids = [d.id for d in docs]
    m = evaluate(ids, demo_rel, k=5)
    print(f'\n[{ret.name}]  MRR={m["mrr"]:.3f}  Recall@5={m[f"recall@5"]:.3f}  ({lat:.1f}ms)')
    for i, d in enumerate(docs[:3], 1):
        flag = 'RELEVANT' if d.id in demo_rel else '        '
        print(f'  {i}. [{flag}] {d.meta["title"]}: {d.content[:70]}...')

## Decision Matrix

In [ ]:
best_by_type = ret_df.groupby(['query_type', 'retriever'])['mrr'].mean()
best_by_diff = ret_df.groupby(['difficulty', 'retriever'])['mrr'].mean()

print('BEST RETRIEVER BY QUERY TYPE')
print('=' * 55)
for qt in sorted(ret_df['query_type'].unique()):
    best = best_by_type[qt].idxmax()
    score = best_by_type[qt].max()
    print(f'  {qt:<14s}: {best:<22s} (MRR={score:.4f})')

print('\nBEST RETRIEVER BY DIFFICULTY')
print('=' * 55)
for diff in ['easy', 'medium', 'hard']:
    if diff in best_by_diff.index:
        best = best_by_diff[diff].idxmax()
        score = best_by_diff[diff].max()
        print(f'  {diff:<10s}: {best:<22s} (MRR={score:.4f})')

print('\nLATENCY BUDGET RECOMMENDATIONS')
print('=' * 55)
lat_order = ret_summary.sort_values('latency')
print(f'  Tightest latency            : {lat_order.index[0]}')
print(f'  Best quality, no constraint : Hybrid RRF + Cross-Encoder')

## Summary & Key Findings

### Retriever Comparison

| Strategy | Strength | Weakness | Best For |
|---|---|---|---|
| **BM25** | Fast, exact match | Vocabulary mismatch | Keyword / factual |
| **TF-IDF Cosine** | Simple, interpretable | No semantics | Short exact queries |
| **Dense Embedding** | Semantic understanding | Compute cost | Conceptual / paraphrased |
| **ColBERT MaxSim** | Best per-token coverage | Slowest | Complex multi-concept |
| **Hybrid RRF** | Best of sparse+dense | Higher latency | Mixed / unknown type |
| **Hybrid Linear** | Tunable balance | Score calibration needed | Known score distributions |

### Reranker Comparison

| Strategy | Strength | Weakness | Best For |
|---|---|---|---|
| **Cross-Encoder** | Highest precision | Slow O(n) passes | High-stakes, low QPS |
| **MonoT5** | Good balance | Needs T5 deployed | When you already have T5 |
| **Cohere-style API** | No GPU needed | API cost and latency | Production with budget |
| **MMR Diversity** | Reduces redundancy | Can reduce MRR | Multi-document summaries |

### The Two-Stage Principle

```
Single-stage:  Query --> [Accurate slow model @ 1000 QPS]  -- too expensive

Two-stage:     Query --> [Fast BM25/ANN, top-50]           -- cheap, high recall
                      --> [Cross-encoder, top-5]           -- expensive but only 50 pairs
```

### Production Haystack Pipeline

```python
from haystack import Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import (
    InMemoryBM25Retriever, InMemoryEmbeddingRetriever
)
from haystack.components.joiners import DocumentJoiner
from haystack.components.rankers import TransformersSimilarityRanker

pipe = Pipeline()
pipe.add_component('bm25',     InMemoryBM25Retriever(store, top_k=30))
pipe.add_component('dense',    InMemoryEmbeddingRetriever(store, top_k=30))
pipe.add_component('joiner',   DocumentJoiner(join_mode='reciprocal_rank_fusion'))
pipe.add_component('reranker', TransformersSimilarityRanker(top_k=5))
pipe.connect('bm25.documents',   'joiner.documents')
pipe.connect('dense.documents',  'joiner.documents')
pipe.connect('joiner.documents', 'reranker.documents')
```

### Production LangChain Pipeline

```python
from langchain.retrievers import EnsembleRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

ensemble = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever], weights=[0.5, 0.5]
)
reranker = CrossEncoderReranker(
    model=HuggingFaceCrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2'),
    top_n=5
)
final = ContextualCompressionRetriever(
    base_retriever=ensemble, doc_compressor=reranker
)
```

---
*Educational notebook -- NLP / Information Retrieval Portfolio.*